In [23]:
from pathlib import Path
import pandas as pd
import numpy as np

# Move one level up from preprocessing to the project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw"
TRAIN_DIR = DATA_ROOT / "train"
STUDENT_TEST_DIR = DATA_ROOT / "student_test"

DATA_ROOT = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_ROOT / "train"
STUDENT_TEST_DIR = DATA_ROOT / "student_test"

print("Project root:", PROJECT_ROOT)
print("Training trips:", len(list(TRAIN_DIR.iterdir())))
print("Student testing trips:", len(list(STUDENT_TEST_DIR.iterdir())))

Project root: d:\Sem 5\Machine Learning\JOB\Jackfruit\PathSnap
Training trips: 80
Student testing trips: 10


In [2]:
trip_id = "00000001"
trip_dir = TRAIN_DIR / trip_id

track = pd.read_csv(
    trip_dir / f"{trip_id}.track",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude", "timestamp"]
)

nodes = pd.read_csv(
    trip_dir / f"{trip_id}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

arcs = pd.read_csv(
    trip_dir / f"{trip_id}.arcs",
    sep=r"\s+",
    header=None,
    names=["from_node_id", "to_node_id"]
)

route = pd.read_csv(
    trip_dir / f"{trip_id}.route",
    header=None,
    names=["arc_id"]
)

print("TRACK")
display(track.head())

print("NODES")
display(nodes.head())

print("ARCS")
display(arcs.head())

print("ROUTE")
display(route.head())

TRACK


,longitude,latitude,timestamp
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0


NODES


,longitude,latitude
0,8.581679,49.857880
1,8.584637,49.858295
2,8.580173,49.858567
3,8.584392,49.859119
4,8.584935,49.857396


ARCS


,from_node_id,to_node_id
0,0,281
1,0,1
2,0,5
3,1,0
4,1,40750


ROUTE


,arc_id
0,2641
1,64973
2,133948
3,133949
4,139208


In [3]:
# Basic dimensions
print("TRACK shape:", track.shape)
print("NODES shape:", nodes.shape)
print("ARCS shape:", arcs.shape)
print("ROUTE shape:", route.shape)

print("\nDATA TYPES")
print("\nTRACK:")
print(track.dtypes)

print("\nNODES:")
print(nodes.dtypes)

print("\nARCS:")
print(arcs.dtypes)

print("\nROUTE:")
print(route.dtypes)

# Missing values
print("\nMISSING VALUES")
print("TRACK:\n", track.isnull().sum())
print("NODES:\n", nodes.isnull().sum())
print("ARCS:\n", arcs.isnull().sum())
print("ROUTE:\n", route.isnull().sum())

TRACK shape: (1315, 3)
NODES shape: (88809, 2)
ARCS shape: (165790, 2)
ROUTE shape: (525, 1)

DATA TYPES

TRACK:
longitude    float64
latitude     float64
timestamp    float64
dtype: object

NODES:
longitude    float64
latitude     float64
dtype: object

ARCS:
from_node_id    int64
to_node_id      int64
dtype: object

ROUTE:
arc_id    int64
dtype: object

MISSING VALUES
TRACK:
 longitude    0
latitude     0
timestamp    0
dtype: int64
NODES:
 longitude    0
latitude     0
dtype: int64
ARCS:
 from_node_id    0
to_node_id      0
dtype: int64
ROUTE:
 arc_id    0
dtype: int64


In [4]:
# Coordinate ranges
print("GPS coordinate ranges:")
print(track[["longitude", "latitude"]].agg(["min", "max"]))

print("\nRoad node coordinate ranges:")
print(nodes[["longitude", "latitude"]].agg(["min", "max"]))

# Timestamp inspection
print("\nTimestamp statistics:")
print(track["timestamp"].describe())

print("\nTimestamp differences:")
print(track["timestamp"].diff().value_counts().head())

# Check road-network ID validity
print("\nMaximum node ID in ARCS:",
      arcs[["from_node_id", "to_node_id"]].max().max())

print("Number of nodes:", len(nodes))

print("\nMaximum arc ID in ROUTE:", route["arc_id"].max())
print("Number of arcs:", len(arcs))

GPS coordinate ranges:
     longitude   latitude
min   8.570524  49.844053
max   8.647860  50.211882

Road node coordinate ranges:
     longitude   latitude
min   8.560594  49.833564
max   8.657860  50.221877

Timestamp statistics:
count    1315.000000
mean      661.000760
std       384.387699
min         1.000000
25%       329.500000
50%       658.000000
75%       988.500000
max      1358.000000
Name: timestamp, dtype: float64

Timestamp differences:
timestamp
1.0     1289
2.0       21
3.0        2
16.0       1
4.0        1
Name: count, dtype: int64

Maximum node ID in ARCS: 88808
Number of nodes: 88809

Maximum arc ID in ROUTE: 165275
Number of arcs: 165790


In [5]:
# 1. Missing values
print("MISSING VALUES")
for name, df in {
    "TRACK": track,
    "NODES": nodes,
    "ARCS": arcs,
    "ROUTE": route
}.items():
    print(name, ":", df.isnull().sum().to_dict())


# 2. Invalid coordinate values
print("\nINVALID COORDINATES")

print("GPS non-finite coordinates:",
      (~np.isfinite(track[["longitude", "latitude"]])).any(axis=1).sum())

print("Node non-finite coordinates:",
      (~np.isfinite(nodes[["longitude", "latitude"]])).any(axis=1).sum())


# 3. Timestamp checks
print("\nTIMESTAMP CHECKS")

print("Timestamps sorted:",
      track["timestamp"].is_monotonic_increasing)

print("Duplicate timestamps:",
      track["timestamp"].duplicated().sum())

print("All timestamp intervals:")
print(track["timestamp"].diff().value_counts().sort_index())


# 4. Road-network ID validation
print("\nROAD NETWORK VALIDATION")

invalid_from = (~arcs["from_node_id"].between(0, len(nodes) - 1)).sum()
invalid_to = (~arcs["to_node_id"].between(0, len(nodes) - 1)).sum()

invalid_route = (~route["arc_id"].between(0, len(arcs) - 1)).sum()

print("Invalid from_node_id:", invalid_from)
print("Invalid to_node_id:", invalid_to)
print("Invalid route arc_id:", invalid_route)


# 5. Duplicate connections
print("\nDUPLICATE ARCS:",
      arcs.duplicated().sum())

MISSING VALUES
TRACK : {'longitude': 0, 'latitude': 0, 'timestamp': 0}
NODES : {'longitude': 0, 'latitude': 0}
ARCS : {'from_node_id': 0, 'to_node_id': 0}
ROUTE : {'arc_id': 0}

INVALID COORDINATES
GPS non-finite coordinates: 0
Node non-finite coordinates: 0

TIMESTAMP CHECKS
Timestamps sorted: True
Duplicate timestamps: 0
All timestamp intervals:
timestamp
1.0     1289
2.0       21
3.0        2
4.0        1
16.0       1
Name: count, dtype: int64

ROAD NETWORK VALIDATION
Invalid from_node_id: 0
Invalid to_node_id: 0
Invalid route arc_id: 0

DUPLICATE ARCS: 292


In [6]:
# Exact duplicate connections
duplicate_rows = arcs[arcs.duplicated(keep=False)]

print("Total exact duplicate rows:", arcs.duplicated().sum())
print("Rows involved in duplicate connections:", len(duplicate_rows))

# Number of unique directed connections
print("Unique directed connections:", len(arcs.drop_duplicates()))

# Check whether reverse-direction connections exist
arc_pairs = set(zip(arcs["from_node_id"], arcs["to_node_id"]))

reverse_pairs = sum(
    1 for u, v in arc_pairs
    if (v, u) in arc_pairs and u != v
)

print("Directed connections with a reverse connection:", reverse_pairs)

print("\nExample duplicate connections:")
display(duplicate_rows.head(10))

Total exact duplicate rows: 292
Rows involved in duplicate connections: 584
Unique directed connections: 165498
Directed connections with a reverse connection: 135460

Example duplicate connections:


,from_node_id,to_node_id
8628,5073,28089
8629,5073,5074
8631,5073,28089
8632,5073,5074
8638,5076,28925
8639,5076,28925
8641,5077,33427
8642,5077,33427
9493,5467,39277
9496,5467,39277


In [7]:
# Convert the road network into an indexed lookup
arc_lookup = arcs.reset_index(names="arc_id")

# Get the arcs used by the reference route
route_arcs = arc_lookup.iloc[route["arc_id"].to_numpy()].reset_index(drop=True)

# Compare the ending node of each arc with the starting node of the next
continuity = (
    route_arcs["to_node_id"].iloc[:-1].to_numpy()
    == route_arcs["from_node_id"].iloc[1:].to_numpy()
)

print("Total route transitions:", len(continuity))
print("Continuous transitions:", continuity.sum())
print("Discontinuous transitions:", (~continuity).sum())

print("Route continuity percentage:",
      round(continuity.mean() * 100, 2), "%")

Total route transitions: 524
Continuous transitions: 524
Discontinuous transitions: 0
Route continuity percentage: 100.0 %


In [8]:
def explore_trip(trip_dir):
    trip_id = trip_dir.name

    track = pd.read_csv(
        trip_dir / f"{trip_id}.track",
        sep=r"\s+", header=None,
        names=["longitude", "latitude", "timestamp"]
    )

    nodes = pd.read_csv(
        trip_dir / f"{trip_id}.nodes",
        sep=r"\s+", header=None,
        names=["longitude", "latitude"]
    )

    arcs = pd.read_csv(
        trip_dir / f"{trip_id}.arcs",
        sep=r"\s+", header=None,
        names=["from_node_id", "to_node_id"]
    )

    route = pd.read_csv(
        trip_dir / f"{trip_id}.route",
        header=None, names=["arc_id"]
    )

    # Route continuity
    route_ids = route["arc_id"].to_numpy()
    route_arcs = arcs.iloc[route_ids]

    if len(route_arcs) > 1:
        continuity = (
            route_arcs["to_node_id"].to_numpy()[:-1]
            == route_arcs["from_node_id"].to_numpy()[1:]
        )
        continuity_pct = continuity.mean() * 100
    else:
        continuity_pct = np.nan

    # Timestamp intervals
    intervals = track["timestamp"].diff().dropna()

    return {
        "trip_id": trip_id,
        "gps_observations": len(track),
        "road_nodes": len(nodes),
        "road_arcs": len(arcs),
        "route_length": len(route),
        "missing_values": (
            track.isnull().sum().sum()
            + nodes.isnull().sum().sum()
            + arcs.isnull().sum().sum()
            + route.isnull().sum().sum()
        ),
        "duplicate_arcs": arcs.duplicated().sum(),
        "timestamp_gaps": (intervals > 1).sum(),
        "invalid_node_ids": (
            (~arcs["from_node_id"].between(0, len(nodes)-1)).sum()
            + (~arcs["to_node_id"].between(0, len(nodes)-1)).sum()
        ),
        "invalid_route_ids": (
            (~route["arc_id"].between(0, len(arcs)-1)).sum()
        ),
        "route_continuity_pct": continuity_pct
    }


# Explore all training trips
trip_folders = sorted(
    [p for p in TRAIN_DIR.iterdir() if p.is_dir()]
)

results = []

for folder in trip_folders:
    results.append(explore_trip(folder))

eda_summary = pd.DataFrame(results)

display(eda_summary)

,trip_id,gps_observations,road_nodes,road_arcs,route_length,missing_values,duplicate_arcs,timestamp_gaps,invalid_node_ids,invalid_route_ids,route_continuity_pct
0,00000001,1315,88809,165790,525,0,292,25,0,0,100.000000
1,00000002,7243,7936,16139,262,0,0,184,0,0,99.616858
2,00000003,821,9138,18028,79,0,6,3,0,0,100.000000
3,00000004,1064,7350,15499,320,0,5,491,0,0,100.000000
4,00000005,1045,898,1579,105,0,0,0,0,0,100.000000
...,...,...,...,...,...,...,...,...,...,...,...
75,00000095,1551,1888,3723,306,0,0,5,0,0,100.000000
76,00000096,3271,16893,32276,456,0,3,0,0,0,100.000000
77,00000097,1740,23566,48503,444,0,0,14,0,0,100.000000
78,00000098,5584,43440,88137,1003,0,10,1268,0,0,100.000000


In [9]:
print("TRAINING DATASET SUMMARY")
print("Number of trips:", len(eda_summary))

display(
    eda_summary[
        [
            "gps_observations",
            "road_nodes",
            "road_arcs",
            "route_length",
            "duplicate_arcs",
            "timestamp_gaps",
            "route_continuity_pct"
        ]
    ].describe().round(2)
)

print("\nQUALITY CHECKS")
print("Trips with missing values:",
      (eda_summary["missing_values"] > 0).sum())

print("Trips with invalid node IDs:",
      (eda_summary["invalid_node_ids"] > 0).sum())

print("Trips with invalid route IDs:",
      (eda_summary["invalid_route_ids"] > 0).sum())

print("Trips with imperfect route continuity:",
      (eda_summary["route_continuity_pct"] < 100).sum())

TRAINING DATASET SUMMARY
Number of trips: 80


,gps_observations,road_nodes,road_arcs,route_length,duplicate_arcs,timestamp_gaps,route_continuity_pct
count,80.00,80.00,80.00,80.00,80.00,80.00,80.00
mean,2605.21,24168.56,48461.30,391.70,22.19,91.08,99.93
std,3312.35,32272.30,64738.84,361.17,54.71,206.79,0.25
min,361.00,299.00,649.00,59.00,0.00,0.00,98.13
25%,945.75,3487.75,7061.75,168.25,0.00,0.75,100.00
50%,1529.00,9608.00,19526.50,280.00,2.00,15.50,100.00
75%,2792.75,28846.75,58003.25,480.50,12.00,69.75,100.00
max,19313.00,155003.00,307392.00,2266.00,322.00,1268.00,100.00



QUALITY CHECKS
Trips with missing values: 0
Trips with invalid node IDs: 0
Trips with invalid route IDs: 0
Trips with imperfect route continuity: 10


In [10]:
# Find trips with imperfect route continuity
problem_trips = eda_summary[
    eda_summary["route_continuity_pct"] < 100
]

display(
    problem_trips[
        ["trip_id", "route_length", "route_continuity_pct"]
    ]
)

,trip_id,route_length,route_continuity_pct
1,00000002,262,99.616858
14,00000019,545,99.448529
26,00000033,169,99.404762
33,00000041,340,99.705015
34,00000042,1440,99.930507
55,00000069,467,99.785408
60,00000075,108,98.130841
63,00000080,139,99.275362
69,00000086,671,99.850746
71,00000089,265,99.621212


In [11]:
# Inspect the exact discontinuities
for trip_id in problem_trips["trip_id"]:

    trip_dir = TRAIN_DIR / trip_id

    arcs = pd.read_csv(
        trip_dir / f"{trip_id}.arcs",
        sep=r"\s+",
        header=None,
        names=["from_node_id", "to_node_id"]
    )

    route = pd.read_csv(
        trip_dir / f"{trip_id}.route",
        header=None,
        names=["arc_id"]
    )

    route_arcs = arcs.iloc[route["arc_id"].to_numpy()].reset_index(drop=True)

    connected = (
        route_arcs["to_node_id"].to_numpy()[:-1]
        == route_arcs["from_node_id"].to_numpy()[1:]
    )

    broken_positions = np.where(~connected)[0]

    print(f"\nTrip: {trip_id}")
    print("Discontinuous transitions:", len(broken_positions))

    for pos in broken_positions[:5]:
        print(
            f"Arc {route.iloc[pos, 0]} ends at "
            f"{route_arcs.iloc[pos]['to_node_id']}; "
            f"next arc {route.iloc[pos + 1, 0]} starts at "
            f"{route_arcs.iloc[pos + 1]['from_node_id']}"
        )


Trip: 00000002
Discontinuous transitions: 1
Arc 1848 ends at 5059; next arc 10863 starts at 5289

Trip: 00000019
Discontinuous transitions: 3
Arc 85589 ends at 849; next arc 51027 starts at 23488
Arc 55059 ends at 25413; next arc 55050 starts at 25408
Arc 953 ends at 10107; next arc 23709 starts at 10862

Trip: 00000033
Discontinuous transitions: 1
Arc 1003 ends at 1283; next arc 1002 starts at 563

Trip: 00000041
Discontinuous transitions: 1
Arc 1264 ends at 349; next arc 519 starts at 348

Trip: 00000042
Discontinuous transitions: 1
Arc 14661 ends at 1762; next arc 17989 starts at 8726

Trip: 00000069
Discontinuous transitions: 1
Arc 10165 ends at 5083; next arc 10172 starts at 5084

Trip: 00000075
Discontinuous transitions: 2
Arc 507 ends at 250; next arc 509 starts at 227
Arc 134 ends at 229; next arc 552 starts at 247

Trip: 00000080
Discontinuous transitions: 1
Arc 8899 ends at 4222; next arc 9194 starts at 4363

Trip: 00000086
Discontinuous transitions: 1
Arc 230068 ends at 401

In [12]:
def inspect_discontinuity(trip_id, first_arc_id, second_arc_id):

    trip_dir = TRAIN_DIR / trip_id

    nodes = pd.read_csv(
        trip_dir / f"{trip_id}.nodes",
        sep=r"\s+",
        header=None,
        names=["longitude", "latitude"]
    )

    arcs = pd.read_csv(
        trip_dir / f"{trip_id}.arcs",
        sep=r"\s+",
        header=None,
        names=["from_node_id", "to_node_id"]
    )

    for arc_id in [first_arc_id, second_arc_id]:

        arc = arcs.iloc[arc_id]

        start = nodes.iloc[arc["from_node_id"]]
        end = nodes.iloc[arc["to_node_id"]]

        print(f"\nArc ID: {arc_id}")
        print(f"From node: {arc['from_node_id']}")
        print(f"To node: {arc['to_node_id']}")
        print(
            f"Start coordinates: "
            f"{start['longitude']}, {start['latitude']}"
        )
        print(
            f"End coordinates: "
            f"{end['longitude']}, {end['latitude']}"
        )


inspect_discontinuity("00000002", 1848, 10863)


Arc ID: 1848
From node: 846
To node: 5059
Start coordinates: 11.046528, 47.664571
End coordinates: 11.0465, 47.664618

Arc ID: 10863
From node: 5289
To node: 5290
Start coordinates: 11.047042, 47.667515
End coordinates: 11.046852, 47.667648


In [31]:
import networkx as nx

# Explicitly reload trip 00000002
trip_id = "00000002"
trip_dir = TRAIN_DIR / trip_id

arcs_trip2 = pd.read_csv(
    trip_dir / f"{trip_id}.arcs",
    sep=r"\s+",
    header=None,
    names=["from_node_id", "to_node_id"]
)

G = nx.from_pandas_edgelist(
    arcs_trip2,
    source="from_node_id",
    target="to_node_id",
    create_using=nx.DiGraph()
)

start_node = 5059
end_node = 5289

if nx.has_path(G, start_node, end_node):
    path = nx.shortest_path(G, start_node, end_node)
    print("Connecting path exists!")
    print("Number of nodes:", len(path))
    print(path)
else:
    print("No directed connecting path exists.")

Connecting path exists!
Number of nodes: 42
[5059, 5058, 4650, 5056, 4651, 4707, 4706, 4704, 6820, 22, 4732, 4962, 4733, 4963, 4841, 552, 4951, 23, 4947, 5175, 5176, 377, 1135, 5179, 722, 369, 723, 368, 721, 1134, 720, 1133, 1132, 719, 7732, 1131, 5284, 5285, 5286, 5287, 5290, 5289]


In [28]:
import folium

center_lat = track["latitude"].mean()
center_lon = track["longitude"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles=None
)

# Esri street map tiles
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, USGS, EPA, USDA, NPS",
    name="Esri Street Map"
).add_to(m)

gps_points = list(zip(
    track["latitude"],
    track["longitude"]
))

folium.PolyLine(
    gps_points,
    color="blue",
    weight=4,
    opacity=0.9,
    tooltip="GPS trajectory"
).add_to(m)

folium.Marker(
    gps_points[0],
    tooltip="Start",
    icon=folium.Icon(color="green")
).add_to(m)

folium.Marker(
    gps_points[-1],
    tooltip="End",
    icon=folium.Icon(color="red")
).add_to(m)

folium.LayerControl().add_to(m)

m

In [29]:
from pathlib import Path

VIS_DIR = PROJECT_ROOT / "visualisations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

print("Visualisations folder:", VIS_DIR)

m.save(str(VIS_DIR / "gps_trajectory_00000001.html"))
print("Map saved successfully!")

Visualisations folder: d:\Sem 5\Machine Learning\JOB\Jackfruit\PathSnap\visualisations
Map saved successfully!


In [30]:
m.save(str(VIS_DIR / "gps_trajectory_00000001.html"))

In [22]:
eda_summary.to_csv(
    VIS_DIR / "training_eda_summary.csv",
    index=False
)

print("EDA summary saved!")

EDA summary saved!


In [24]:
problem_trips.to_csv(
    VIS_DIR / "route_discontinuities_summary.csv",
    index=False
)

print("Route discontinuity summary saved!")

Route discontinuity summary saved!
